# Part 3: Introduction to Statistics
**⏱ This section takes approximately 30 minutes.**

## Scenario: Did the Email Campaign Actually Work?

Your marketing team ran a new email campaign. The open rate went from **22% to 28%**.

> "That's a 6-percentage-point improvement! Huge win!" — Marketing team

But you're the data analyst. Before celebrating, you need to answer:
**Is this a real improvement, or could it just be random variation?**

Z-scores, p-values, and hypothesis testing are the tools that separate genuine signal from noise.

**By the end of this section you'll be able to:**
- Calculate and interpret z-scores in a business context
- Explain what a p-value means in plain English
- Run a hypothesis test in Python and write a business conclusion

In [ ]:
import numpy as np
import scipy.stats as st
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
np.random.seed(42)
sns.set_style('whitegrid')
print("✅ Libraries loaded!")

## 📏 Z-Scores — "How unusual is this result?"

**In plain English:**
A z-score tells you how many standard deviations a value is from the average.

- **Z = 0**: exactly average
- **Z = +2**: unusually high (only ~2.5% of values are this high)
- **Z = -2**: unusually low (only ~2.5% of values are this low)

**Formula:** z = (your_value − mean) ÷ standard_deviation

**Business use:** If the typical email campaign in your industry gets a 22% open rate (σ = 3%), and your campaign got 28%, how unusual is that result? The z-score gives you a precise answer.

In [ ]:
# Scenario: industry email open rates
# Based on historical data: mean = 22%, standard deviation = 3%
industry_mean = 22.0   # percent
industry_std = 3.0     # percent
your_result = 28.0     # percent — your campaign's open rate

# Simulate 10,000 past campaigns to give us a reference distribution
past_campaigns = np.random.normal(industry_mean, industry_std, 10_000)

# Plot the distribution
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(past_campaigns, bins=60, color='lightsteelblue', edgecolor='white', density=True, alpha=0.8)
ax.axvline(industry_mean, color='orange', linewidth=2.5, label=f'Industry mean = {industry_mean}%')
ax.axvline(your_result, color='red', linewidth=2.5, label=f'Your campaign = {your_result}%')
ax.set_xlabel('Email open rate (%)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Industry email open rates — where does your result sit?', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Calculate z-score: how many standard deviations above the mean is your result?
z_score = (your_result - industry_mean) / industry_std

print(f"Your campaign:    {your_result}%")
print(f"Industry mean:    {industry_mean}%")
print(f"Industry std dev: {industry_std}%")
print()
print(f"Z-score: ({your_result} − {industry_mean}) ÷ {industry_std} = {z_score:.2f}")
print()
print(f"Interpretation: Your result is {z_score:.1f} standard deviations above the mean.")
print(f"Only {100*(1 - st.norm.cdf(z_score)):.1f}% of campaigns perform this well or better.")


### 💡 What does z = 2.0 mean?

A z-score of 2.0 means your campaign is 2 standard deviations above the average.

Using the **68-95-99.7 rule** from Part 2:
- ~95% of campaigns fall within 2 standard deviations of the mean
- Your result is right at that boundary — only ~2.5% of campaigns do this well

**Is it luck, or is it real?** A high z-score suggests the result is unusual. But "unusual" isn't the same as "definitely real" — that's where p-values come in.

## 🎯 P-Values — "What's the probability this happened by chance?"

**In plain English:**
> A p-value is the probability of seeing a result **this extreme (or more extreme)** if there was actually no real effect.

**The key question it answers:**
"If our old and new campaigns perform equally well (null hypothesis), what's the chance we'd see a difference this large just due to random variation?"

**The 0.05 threshold (α):**
- **p < 0.05**: Less than 5% chance it's just luck → we call it "statistically significant"
- **p ≥ 0.05**: Could easily be random variation → we can't be confident

**Important:** p < 0.05 does NOT mean "100% definitely a real effect". It means "the evidence is strong enough that we'd act on it".

In [ ]:
# Calculate p-value for our email campaign result
# One-tailed: probability of seeing open rate >= 28% by chance
p_one_tailed = 1 - st.norm.cdf(z_score)

# Two-tailed: probability of seeing a result this extreme in EITHER direction
p_two_tailed = 2 * p_one_tailed

print(f"Z-score:           {z_score:.2f}")
print()
print(f"P-value (one-tailed): {p_one_tailed:.4f} = {p_one_tailed*100:.2f}%")
print(f"  → Probability of seeing open rate ≥ {your_result}% by chance alone")
print()
print(f"P-value (two-tailed): {p_two_tailed:.4f} = {p_two_tailed*100:.2f}%")
print(f"  → Probability of seeing a difference this large in either direction")
print()
alpha = 0.05
if p_two_tailed < alpha:
    print(f"✅ p = {p_two_tailed:.4f} < α = {alpha} → STATISTICALLY SIGNIFICANT")
    print(f"   We have enough evidence to say the campaign improvement is real.")
else:
    print(f"⚠️  p = {p_two_tailed:.4f} ≥ α = {alpha} → NOT statistically significant")
    print(f"   We cannot rule out that this improvement is due to random chance.")


### ⏸️ Pause and Predict

What if the campaign had only shown 24% instead of 28%?

**Before running the next cell, predict:**
- Would the z-score be higher or lower?
- Would the p-value be higher or lower?
- Would the result be statistically significant?

*Write your prediction here:*

In [ ]:
# Let's compare: what if the improvement was smaller?
print("Comparing different campaign results:
")
print(f"{'Open Rate':>12} | {'Z-score':>8} | {'P-value':>10} | {'Significant?':>14}")
print("-" * 55)

for rate in [22, 23, 24, 25, 26, 27, 28, 29, 30]:
    z = (rate - industry_mean) / industry_std
    p = 2 * (1 - st.norm.cdf(abs(z)))
    sig = "✅ YES" if p < 0.05 else "❌ no"
    arrow = " ← your result" if rate == your_result else ""
    print(f"{rate:>11}%  | {z:>8.2f} | {p:>10.4f} | {sig:>14}{arrow}")


### 💡 What do you notice?

- Results close to the mean (22–24%) are **not significant** — they could easily happen by chance
- As the open rate increases above 25%, the p-value drops below 0.05 and the result becomes **statistically significant**
- Your 28% result has p < 0.05 → the improvement is unlikely to be just luck

**Important caveat:** Statistical significance doesn't mean *practical* significance. A 23% vs 22% improvement might be significant with a huge dataset, but useless in practice. Always ask: "Is the effect size large enough to matter?"

## 🔬 Hypothesis Testing — The Full Framework

Now let's run a proper hypothesis test comparing **two groups** — the kind you'd run in a real A/B test.

**The scenario:**
- **Group A (control):** 500 users received the old email → recorded their open rates (yes/no)
- **Group B (treatment):** 500 users received the new email → recorded their open rates (yes/no)
- We want to know: is the difference in open rates statistically significant?

**The four steps:**
1. State your hypotheses (H₀ and H₁)
2. Collect data and choose a test
3. Calculate the test statistic and p-value
4. Make a decision

In [ ]:
# Step 1: State hypotheses
print("STEP 1: State Your Hypotheses")
print("=" * 45)
print("H₀ (null):        The two campaigns have equal open rates")
print("H₁ (alternative): The new campaign has a higher open rate")
print()
print("Significance level: α = 0.05")
print("(We accept 5% risk of wrongly claiming an effect exists)")


In [ ]:
# Step 2: Simulate the data (in practice, you'd load a CSV)
np.random.seed(42)
n_per_group = 500

# Group A: old campaign (true open rate = 22%)
group_a = np.random.binomial(1, 0.22, n_per_group)  # 1 = opened, 0 = didn't

# Group B: new campaign (true open rate = 28%)
group_b = np.random.binomial(1, 0.28, n_per_group)

print("STEP 2: Collect & Inspect Data")
print("=" * 45)
print(f"Group A (old campaign): n={n_per_group}, open rate = {group_a.mean():.1%}")
print(f"Group B (new campaign): n={n_per_group}, open rate = {group_b.mean():.1%}")
print(f"Observed difference:    {(group_b.mean() - group_a.mean()):.1%}")


In [ ]:
# Step 3: Run a two-sample t-test to compare means
# The t-test is the right choice here: comparing two group means
t_stat, p_value = st.ttest_ind(group_a, group_b)

print("STEP 3: Run the Statistical Test")
print("=" * 45)
print(f"Test used:     Two-sample independent t-test")
print(f"T-statistic:   {t_stat:.3f}")
print(f"P-value:       {p_value:.4f}")
print()
print("Interpretation of t-statistic:")
print(f"  The difference between groups is {abs(t_stat):.1f} standard errors away from zero.")
print(f"  The larger |t|, the less likely the difference is due to chance.")


In [ ]:
# Step 4: Make the decision
print("STEP 4: Make Your Decision")
print("=" * 45)
alpha = 0.05

if p_value < alpha:
    print(f"✅ p = {p_value:.4f} < α = {alpha}")
    print()
    print("Decision: REJECT H₀")
    print()
    print("Business conclusion (write this for your manager):")
    print(f'  "The new email campaign achieved a {group_b.mean():.1%} open rate vs {group_a.mean():.1%}')
    print(f'   for the old campaign (n=500 per group). The difference is statistically')
    print(f'   significant (p={p_value:.3f}). We recommend rolling out the new campaign."')
else:
    print(f"⚠️  p = {p_value:.4f} ≥ α = {alpha}")
    print()
    print("Decision: FAIL TO REJECT H₀")
    print("  We don't have enough evidence to conclude the difference is real.")


### 💡 A common mistake: "failing to reject" ≠ "proving H₀ is true"

If p ≥ 0.05, it does NOT mean the campaigns perform equally. It means:
- Our sample wasn't large enough to detect the effect, OR
- The effect really is small/zero

This is why "we need more data" is often the right business recommendation when p is close to 0.05.

In [ ]:
# Visualise the results clearly for a non-technical audience
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: Bar chart comparing open rates
groups = ['Old Campaign\n(Group A)', 'New Campaign\n(Group B)']
rates = [group_a.mean() * 100, group_b.mean() * 100]
colors = ['lightsteelblue', 'teal']

bars = axes[0].bar(groups, rates, color=colors, edgecolor='white', width=0.5)
axes[0].set_ylabel('Open Rate (%)', fontsize=12)
axes[0].set_title('Campaign Open Rates', fontsize=13)
axes[0].set_ylim(0, 40)
for bar, rate in zip(bars, rates):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{rate:.1f}%', ha='center', fontsize=13, fontweight='bold')
axes[0].axhline(industry_mean, color='orange', linestyle='--', linewidth=1.5, label=f'Industry avg {industry_mean}%')
axes[0].legend()

# Right: Distribution of individual responses
x_pos = np.random.normal(0, 0.08, n_per_group)
axes[1].scatter(x_pos, group_a + np.random.normal(0, 0.02, n_per_group),
               alpha=0.15, color='lightsteelblue', s=15, label='Group A')
axes[1].scatter(x_pos + 1, group_b + np.random.normal(0, 0.02, n_per_group),
               alpha=0.15, color='teal', s=15, label='Group B')
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(['Group A', 'Group B'])
axes[1].set_yticks([0, 1])
axes[1].set_yticklabels(['Did not open', 'Opened'])
axes[1].set_title(f'Individual responses (p = {p_value:.3f})', fontsize=13)
axes[1].legend(loc='upper right')

plt.suptitle('Email Campaign A/B Test Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## ✅ Section 3 Summary

| Tool | What it tells you | When to use it |
|---|---|---|
| **Z-score** | How unusual a single value is | Comparing one result to a known distribution |
| **P-value** | Probability result is due to chance | Any hypothesis test |
| **T-test** | Whether two group means differ significantly | A/B tests, before/after comparisons |

**The four-step hypothesis testing framework:**
1. State H₀ (no effect) and H₁ (effect exists)
2. Collect data and choose your test
3. Calculate p-value
4. If p < 0.05: reject H₀ and communicate the result clearly

**Back to our email campaign:**
> p = 0.006 < 0.05 → The improvement is statistically significant. The new campaign is genuinely better, not just lucky.

---

## 🏁 Module 3.1 Complete!

You've covered three foundational sections:
- **Part 1:** Probability & Law of Large Numbers
- **Part 2:** Distributions (Uniform, Normal, CLT)
- **Part 3:** Statistics, Z-scores, P-values & Hypothesis Testing

**Next step:** Open `notebooks/assignment.ipynb` to work through the three-tier practice exercises and the full assignment.